In [8]:
import numpy as np
import math
import av
import cv2
import subprocess
import json
import os

def magnitude(vec):
    return math.sqrt(np.sum(vec**2))

def pooling(frame, dim):
    arr = np.array(frame)
    h = (arr.shape[0] // dim) * dim
    w = (arr.shape[1] // dim) * dim
    arr = arr[:h, :w]
    arr = arr.reshape(h // dim, dim, w // dim, dim)
    return arr.mean(axis=(1, 3))

def cosine_similarity(frame1, frame2):
    a = np.array(frame1).flatten().astype(np.float32)
    b = np.array(frame2).flatten().astype(np.float32)
    numerator = np.dot(a, b)
    denominator = magnitude(a) * magnitude(b)
    return numerator / denominator

def read_frames(
    video_path: str,
    start_sec: float,
    end_sec: float,
    threshold: float,
    blocksize: int = 3,
    log_file=None,  # pass an open file to get the | DETECTED output
):
    container = av.open(video_path)
    stream = container.streams.video[0]
    fps = float(stream.average_rate)

    container.seek(int(start_sec * 1_000_000), any_frame=False, backward=True)

    prev = None
    cut_timestamps = []

    for frame in container.decode(stream):
        if frame.time is None:
            continue

        timestamp_sec = float(frame.time)

        if timestamp_sec < start_sec:
            continue
        if timestamp_sec > end_sec:
            break

        reformatted = frame.reformat(width=480, height=270, format="gray")
        img = reformatted.to_ndarray()

        edges = cv2.Canny(img, 50, 100)
        if np.count_nonzero(edges) == 0:
            continue

        pooled = pooling(edges, blocksize)

        if prev is not None:
            dissimilarity = abs(1 - cosine_similarity(pooled, prev))

            # Offset by one frame so timestamp points to the cut frame not the one before
            adjusted = timestamp_sec + (1.0 / fps)
            sec = int(adjusted)
            frame_in_sec = int((adjusted - sec) * fps)

            if log_file is not None:
                log_file.write(f"{sec}:{frame_in_sec}: {dissimilarity:.6f}")
                if dissimilarity > threshold:
                    log_file.write("  | DETECTED")
                log_file.write("\n")

            if dissimilarity > threshold:
                cut_timestamps.append(adjusted)

        prev = pooled

    container.close()
    return cut_timestamps

def generate_keyframes(video_path: str):
    cmd = [
        "ffprobe",
        "-skip_frame", "nokey",
        "-select_streams", "v:0",
        "-show_frames",
        "-show_entries", "frame=best_effort_timestamp_time",
        "-of", "json",
        video_path
    ]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
    data = json.loads(result.stdout)
    return [
        float(f["best_effort_timestamp_time"])
        for f in data.get("frames", [])
        if "best_effort_timestamp_time" in f
    ]

def keyframe_windows(keyframes, radius=1.0):
    windows = [(max(0, k - radius), k + radius) for k in keyframes]
    windows.sort()
    merged = [windows[0]]
    for start, end in windows[1:]:
        last_start, last_end = merged[-1]
        if start <= last_end:
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))
    return merged

def merge_short_scenes(boundaries, min_duration=0.5):
    if len(boundaries) <= 2:
        return boundaries
    merged = [boundaries[0]]
    for t in boundaries[1:]:
        if t - merged[-1] < min_duration:
            continue
        merged.append(t)
    return merged

def detect_and_trim_scenes(
    original_video_path: str,
    threshold: float,
    radius: float = 1.0,
    output_dir: str = "./output_test",
    blocksize: int = 3,
    log_path: str = "frame.txt",  # set to None to skip logging
):
    os.makedirs(output_dir, exist_ok=True)

    print("Capturing key areas...")
    keyframes = generate_keyframes(original_video_path)
    if not keyframes:
        return []
    windows = keyframe_windows(keyframes, radius)
    print(f"Generated {len(windows)} windows")

    all_cut_timestamps = []

    log_file = open(log_path, "w") if log_path else None

    print("Scanning for scene cuts...")
    for i, (start, end) in enumerate(windows):
        print(f"  Window {i+1}/{len(windows)}: {start:.2f}s → {end:.2f}s")
        cuts = read_frames(original_video_path, start, end, threshold, blocksize, log_file=log_file)
        all_cut_timestamps.extend(cuts)

    if log_file:
        log_file.close()

    all_cut_timestamps = sorted(set(all_cut_timestamps))
    print(f"Found {len(all_cut_timestamps)} cuts: {[f'{t:.2f}' for t in all_cut_timestamps]}")

    duration_cmd = [
        "ffprobe", "-i", original_video_path,
        "-show_entries", "format=duration",
        "-v", "quiet", "-of", "csv=p=0"
    ]
    result = subprocess.run(duration_cmd, stdout=subprocess.PIPE, text=True)
    duration = float(result.stdout.strip())

    scene_boundaries = sorted(set([0.0] + all_cut_timestamps + [duration]))
    scene_boundaries = merge_short_scenes(scene_boundaries, min_duration=0.5)

    print("Cutting scenes...")
    cut_points = scene_boundaries[1:-1]
    out_pattern = os.path.join(output_dir, "scene_%04d.mp4")

    cmd = [
        "ffmpeg", "-y",
        "-i", original_video_path,
        "-c:v", "libx264",
        "-c:a", "aac", 
        "-crf", "18",
        "-preset", "fast",
        "-force_key_frames", ",".join(f"{t:.6f}" for t in cut_points),
        "-f", "segment",
        "-segment_times", ",".join(f"{t:.6f}" for t in cut_points),
        "-reset_timestamps", "1",
        out_pattern
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)

    print(f"Scene boundaries: {scene_boundaries}")
    print(f"Cut points being passed to ffmpeg: {cut_points}")
    final_scenes = []
    for i in range(len(scene_boundaries) - 1):
        start = scene_boundaries[i]
        end = scene_boundaries[i + 1]
        out_path = os.path.join(output_dir, f"scene_{i:04d}.mp4")
        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
            final_scenes.append({
                "scene_index": i,
                "start": start,
                "end": end,
                "path": out_path
            })

    print(f"Done — {len(final_scenes)} scenes exported")
    return final_scenes

# --- RUN ---
input_file = "../test/avatar_test.mp4"
scenes = detect_and_trim_scenes(
    original_video_path=input_file,
    threshold=0.78,
    blocksize=5,
    radius=1.0,
    output_dir="./output_test",
    log_path="frame.txt",
)

Capturing key areas...
Generated 2 windows
Scanning for scene cuts...
  Window 1/2: 0.00s → 5.00s
  Window 2/2: 9.97s → 13.51s
Found 4 cuts: ['1.33', '2.71', '2.92', '13.14']
Cutting scenes...
Scene boundaries: [0.0, 1.3341465053763442, 2.710521505376344, 13.137604838709677, 16.76675]
Cut points being passed to ffmpeg: [1.3341465053763442, 2.710521505376344, 13.137604838709677]
Done — 4 scenes exported


In [10]:
import av

video_path = "../test/avatar_test.mp4"

def scan_threshold(video_path):
    container = av.open(video_path)

    i = 0

    frames = container.decode(video=0)
    
    for i in range (container.decode(video=0)):
        prev = 
        if frame.key_frame:
            print(f"Found {i} key frame!")
            i += 1


scan_threshold(video_path)

SyntaxError: invalid syntax (2668839939.py, line 13)